In [0]:
from pyspark.sql.functions import count, sum, col

def product_performance():
    # Prepare the data
    df = spark.table("workspace.silver_layer.events_cleaned")
    
    gold_df = (df.filter(col("event_type") == "purchase")
               .groupBy("product_id", "category_code", "brand")
               .agg(count("event_type").alias("total_purchases"),
                    sum("price").alias("total_revenue")))

    # Ensure the table exists with the IDENTITY column
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_layer")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS workspace.gold_layer.product_performance (
            performance_id BIGINT GENERATED ALWAYS AS IDENTITY,
            product_id INT,
            category_code STRING,
            brand STRING,
            total_purchases LONG,
            total_revenue DOUBLE,
            CONSTRAINT product_pk PRIMARY KEY(performance_id)
        )
    """)

    # Clear old data and Insert new data
    spark.sql("TRUNCATE TABLE workspace.gold_layer.product_performance")

    gold_df.createOrReplaceTempView("temp_gold_results")

    # Insert into the table, specifying columns EXCEPT the identity column
    spark.sql("""
        INSERT INTO workspace.gold_layer.product_performance 
        (product_id, category_code, brand, total_purchases, total_revenue)
        SELECT product_id, category_code, brand, total_purchases, total_revenue 
        FROM temp_gold_results
    """)

product_performance()

In [0]:
from pyspark.sql import functions as f

# 1. Define the source and target paths
silver_table = "workspace.silver_layer.events_cleaned"
gold_table = "workspace.gold_layer.product_performance_metrics"

# 2. Read from Silver (using batch for a Gold aggregation)
silver_df = spark.read.table(silver_table)

# 3. Aggregate data for Product Performance
product_gold_df = (silver_df
    .filter(f.col("price") > 0) # Basic quality filter
    .groupBy("product_id")
    .agg(
        f.count("user_id").alias("total_views"),
        f.sum("price").alias("potential_revenue"),
        f.avg("price").alias("avg_product_price")
    )
)

# 4. Write to the Gold Layer in Unity Catalog
product_gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table)